# 11 Canonical Rename and Path Engine

Deterministic YAML-first canonicalization step.

What this notebook does:
- loads the latest upstream dataset (`review_snapshot_latest`, `inventory_with_text`, `rule_classification`, or `inventory`)
- loads `SCH_fileserver_policy_v2_4.yaml`
- builds canonical target names and paths **only from deterministic rules**
- flags unresolved rows instead of guessing
- applies long-path shortening in policy order
- exports canonicalization outputs for review before any LLM suggestions


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_4.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


In [ ]:
from src.canonicalize import CanonicalizeConfig, canonicalize_dataframe, load_policy

policy = load_policy(POLICY_PATH)
print('Top level folder pattern:', policy['naming_policy']['company_folder']['pattern'])
print('Internal filename template:', policy['naming_policy']['filenames']['internal']['template'])
print('PV type code present:', 'PVS' in policy['naming_policy']['typeid']['encoding']['types'])


In [ ]:
COMPANY_FOLDER_OVERRIDE = ''
ASSET_FOLDER_OVERRIDE = ''
ARCHIVE_YEAR = ''
ASSUME_ACTIVE_ASSETS = True

ABBREVIATIONS = {
    'agreement': 'agr',
    'engineering': 'eng',
    'construction': 'const',
    'commissioning': 'comm',
    'operation': 'ops',
    'operations': 'ops',
    'financial': 'fin',
    'contract': 'cnt',
    'technical': 'tec',
    'permitting': 'perm',
}


In [ ]:
from pathlib import Path


def latest_matching_path(output_dir: Path, prefixes: list[str]) -> Path | None:
    candidates = []
    for prefix in prefixes:
        candidates.extend(sorted(output_dir.glob(f'{prefix}*.parquet')))
        candidates.extend(sorted(output_dir.glob(f'{prefix}*.csv')))
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


INPUT_PATH = latest_matching_path(
    OUTPUT_DIR,
    ['review_snapshot_latest', 'review_snapshot_', 'inventory_with_text_', 'rule_classification_', 'inventory_']
)
print('INPUT_PATH =', INPUT_PATH)


In [ ]:
from pathlib import Path


def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)


def ensure_canonical_input_schema(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    if 'filename' not in work.columns:
        if 'relative_path' in work.columns:
            work['filename'] = work['relative_path'].astype(str).map(lambda x: Path(x).name)
        else:
            work['filename'] = ''

    if 'suffix' not in work.columns:
        work['suffix'] = work['filename'].astype(str).map(lambda x: Path(x).suffix.lower().lstrip('.'))

    if 'ext' not in work.columns:
        work['ext'] = work['suffix']

    if 'description' not in work.columns:
        if 'text_preview' in work.columns:
            work['description'] = work['text_preview'].fillna('').astype(str).str.split().str[:8].str.join('-')
        else:
            work['description'] = ''

    for col in ['typeid', 'phase', 'doc_type', 'date', 'version', 'status', 'company_folder', 'asset_folder', 'owner_or_project', 'project_name', 'location', 'type', 'metric', 'ss']:
        if col not in work.columns:
            work[col] = ''

    return work


source_df = load_table(INPUT_PATH)
source_df = ensure_canonical_input_schema(source_df)
print('Rows:', len(source_df))
preview_cols = [c for c in ['relative_path', 'filename', 'suffix', 'phase', 'doc_type', 'date', 'status'] if c in source_df.columns]
display(source_df[preview_cols].head(20))


In [ ]:
config = CanonicalizeConfig(
    company_folder_override=COMPANY_FOLDER_OVERRIDE or None,
    asset_folder_override=ASSET_FOLDER_OVERRIDE or None,
    archive_year=ARCHIVE_YEAR or None,
    assume_active_assets=ASSUME_ACTIVE_ASSETS,
    abbreviations=ABBREVIATIONS,
)

canonical = canonicalize_dataframe(source_df, policy, config=config)
print('Rows:', len(canonical))


In [ ]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(canonical, [
    'relative_path', 'canonical_ready', 'canonical_reason', 'unresolved_fields',
    'canonical_relative_path', 'length_actions', 'is_within_limits'
], 20)

display(canonical['canonical_reason'].fillna('missing').value_counts().rename_axis('canonical_reason').reset_index(name='count'))
display(canonical['canonical_ready'].fillna(False).value_counts().rename_axis('canonical_ready').reset_index(name='count'))


In [ ]:
ready_mask = canonical['canonical_ready'].fillna(False)
review_mask = ~ready_mask

_show(canonical.loc[ready_mask], [
    'relative_path', 'canonical_relative_path', 'canonical_filename',
    'length_actions', 'full_path_len', 'filename_len'
], 50)

_show(canonical.loc[review_mask], [
    'relative_path', 'unresolved_fields', 'canonical_notes', 'canonical_reason'
], 50)


In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
parquet_path = OUTPUT_DIR / f'canonical_candidates_{ts}.parquet'
csv_path = OUTPUT_DIR / f'canonical_candidates_{ts}.csv'
canonical.to_parquet(parquet_path, index=False)
canonical.to_csv(csv_path, index=False)

print('Wrote:')
print(parquet_path)
print(csv_path)
